# S0 pequeño, datos diversos: CINIC + DomainNet + AdaIN

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/s0_domain_generalization.ipynb)

**Pregunta:** ¿mejora la generalización de un clasificador de 11,4 M de parámetros si aumentamos la diversidad visual de los datos, manteniendo fijo el presupuesto de aprendizaje?

Este es un experimento nuevo de tres variantes. No modifica el notebook CINIC ejecutado ni continúa sus checkpoints. No usa PE como profesor ni destilación adicional. El preentrenamiento original de MobileCLIP2 sí utilizó profesores.

| Variante | Imágenes de entrenamiento | Diferencia controlada |
|---|---|---|
| A · CINIC | CINIC train | Referencia del nuevo presupuesto |
| B · +DomainNet | CINIC + clipart/painting compatibles | Nuevas fuentes y representaciones |
| C · +AdaIN | Mismos originales y orden que B; 30% de vistas estilizadas | Transferencia de estilo durante entrenamiento |

**Ejecutar:** Colab con GPU, preferiblemente A100 → ejecutar en orden → autorizar Drive. Kaggle usa el secreto `KAGGLE_API_TOKEN`. No requiere token HF.

**Coste:** la primera descarga de DomainNet transfiere aproximadamente 7,5 GB; procesa un ZIP cada vez y conserva solo las categorías compatibles a 32×32. Reserva unos 6 GB libres en el disco temporal y varios GB en Drive para cachés/checkpoints. Los tiempos reales dependen de la GPU y del servidor; hay pausas y reanudación.

**Reglas del estudio:** todo Warehouse excluido. Sin adaptación ni métricas de test. Los resultados externos no predicen numéricamente Kaggle. Exportación final desactivada por defecto.

## 1. Diseño fijado antes de entrenar

- Mismo checkpoint público S0, misma inicialización de cabeza, batch y optimizador.
- **15 épocas × 400 actualizaciones = 6.000 pasos por variante**, sin parada temprana. Dos épocas de cabeza; trece de ajuste completo con LR pequeño y estadísticas BatchNorm congeladas.
- 128 imágenes por paso por defecto. Una época es un presupuesto de muestreo, no un recorrido exhaustivo por el dataset.
- Etiquetas aproximadamente uniformes entre las diez clases. En B/C, para las nueve clases cubiertas, probabilidad 30% de seleccionar DomainNet y elección uniforme de clipart/painting. Los ciervos proceden de CINIC: la ausencia de ciervos de DomainNet queda documentada, no se inventan etiquetas.
- B y C ven exactamente los mismos IDs originales en el mismo orden. C reemplaza aleatoriamente el 30% por su versión AdaIN; todas las clases pueden estilizarse.
- Selección de época y variante por la **media de accuracy macro por dominio**: CINIC, clipart y painting. CINIC cubre diez clases; los otros, nueve. No es una media sobre una distribución natural conocida.
- **Sketch reservado** y validación de la competición se evalúan después de congelar la elección. La validación de la competición ya apareció en experimentos anteriores: no es un conjunto nuevo y no presumimos su origen.
- Se eliminan duplicados RGB exactos y reflejos horizontales entre particiones antes de transformar. No equivale a detectar todos los duplicados aproximados.

La referencia A no reproduce las 25 épocas anteriores: la comparación válida es A/B/C dentro de este notebook. Una sola semilla es una exploración inicial; una mejora pequeña necesitaría repeticiones.

In [ ]:
%pip -q install open_clip_torch==3.2.0 timm==1.0.22 huggingface_hub==0.36.0 kagglehub==1.0.3 pyarrow==21.0.0 requests tqdm pandas matplotlib
# Conservamos la pareja torch/torchvision proporcionada por Colab.


In [ ]:
from pathlib import Path
import sys, hashlib, json, importlib, requests
CODE_REVISION = "e7581f203e38e713bb3d62b398771ab45422c88b"
FILE_BLOBS = {
    "ADAIN_LICENSE.txt": "82c7b41af6e01f19c7aa154c8a8f0fae72eddb96",
    "adain_networks.py": "7bc2939e2d1b66a765363f8e29291f5d73e99ef4",
    "mobileclip_s0_cinic.py": "5017c5e82e3f8b74ad25faf5afd876b6d31e3d34",
    "s0_domain_generalization.py": "8d54ed710e29de27903746610a0e79c474357d23",
    "test_s0_domain_generalization.py": "6818637e8f95a84bbd852ead6560ec429f24b613"
}
CODE_DIR = Path('/content/recvis_domain_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
CODE_HASHES = {}
for filename, expected_blob in FILE_BLOBS.items():
    response = requests.get(
        f'https://raw.githubusercontent.com/rivas-vega/A3_Recv/{CODE_REVISION}/{filename}', timeout=90)
    response.raise_for_status()
    raw_code = response.content
    git_blob = hashlib.sha1(b'blob ' + str(len(raw_code)).encode() + b'\0' + raw_code).hexdigest()
    if git_blob != expected_blob:
        raise RuntimeError(f'Código distinto de la revisión fijada: {filename}')
    (CODE_DIR / filename).write_bytes(raw_code)
    CODE_HASHES[filename] = hashlib.sha256(raw_code).hexdigest()
if str(CODE_DIR) not in sys.path: sys.path.insert(0, str(CODE_DIR))
for module in ('s0_domain_generalization', 'mobileclip_s0_cinic', 'adain_networks'):
    sys.modules.pop(module, None)
importlib.invalidate_caches()
import mobileclip_s0_cinic as base
import s0_domain_generalization as m
print('Código verificado:', CODE_REVISION)


In [ ]:
import os, time, gc, platform, importlib.metadata
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
RUN_NAME = 's0_domain_generalization_v1'
DRIVE_ROOT = Path('/content/drive/MyDrive/A3_Recv')
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
CACHE_DIR = DRIVE_ROOT / 'cache_s0_domain_v1'
STYLE_DIR = RUN_DIR / 'style_cache'
for path in (RUN_DIR, CACHE_DIR): path.mkdir(parents=True, exist_ok=True)

CFG = dict(m.DEFAULTS)
SESSION_HOURS = 6.5  # presupuesto de la celda de entrenamiento, compartido por A/B/C
STYLE_HOURS = 1.5    # presupuesto por llamada para crear la caché, se ejecuta por separado
EXPORT_FINAL = False
base.lock_json(CFG, RUN_DIR / 'config.json')
if not torch.cuda.is_available(): raise RuntimeError('Selecciona un entorno de ejecución con GPU.')
print('GPU real:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
print('Destino:', RUN_DIR)
print('Pasos por variante:', CFG['max_epochs'] * CFG['steps_per_epoch'])
VERSIONS = {p: importlib.metadata.version(p) for p in
            ('torch', 'torchvision', 'open_clip_torch', 'timm', 'numpy', 'pillow')}
base.lock_json(dict(code_revision=CODE_REVISION, code_sha256=CODE_HASHES,
                    packages=VERSIONS, python=platform.python_version()),
               RUN_DIR / 'environment.json')
# Si cambia la configuración, el código o las versiones, usa un RUN_NAME nuevo.


## 2. Comprobaciones rápidas y acceso a los datos

Las pruebas rápidas usan imágenes sintéticas pequeñas. Comprueban la separación de particiones, etiquetas, muestreo pareado B/C, estabilidad de la configuración y elección final.

La credencial de Kaggle se guarda una vez en **Secretos (llave) → KAGGLE_API_TOKEN → permitir acceso**. No se pide en un output ni se imprime. Se admite también el par antiguo KAGGLE_USERNAME/KAGGLE_KEY.

La descarga puede contener los archivos de la competición, pero esta sección solo abre **validation.npz** para excluir coincidencias de entrenamiento. Sus métricas se reservan para el final.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', '-q',
                'test_s0_domain_generalization.ProtocolTests'],
               cwd=CODE_DIR, check=True)
base.configure_kaggle_secrets()
import kagglehub
competition_root = Path(kagglehub.competition_download('mva-recvis-2026'))
audit_x, audit_y, audit_sha = base.read_validation(competition_root)
print('Validación etiquetada cargada para separación:', len(audit_y))


## 3. CINIC y DomainNet

CINIC se obtiene del mirror público ya verificado de Flower en Hugging Face. Se usan train/valid; CINIC test queda fuera.

DomainNet procede de los ZIP oficiales y sus listados depurados, fijados por SHA256. Solo se conservan imágenes de categorías compatibles. `airplane, car, bird, cat, dog, frog, horse, truck` se mapean explícitamente, y `aircraft_carrier, cruise_ship, sailboat, speedboat` se agrupan como `ship`. No se agrupan automáticamente categorías ambiguas.

Los listados comprobados contienen **1.221 clipart-train, 2.173 painting-train, 532 clipart-test, 944 painting-test y 853 sketch-test** compatibles, antes de deduplicar. Las cifras finales se imprimen abajo.

**Importante:** unas 3.400 imágenes nuevas frente a decenas de miles de CINIC requieren muestreo explícito para tener presencia. La sobremuestra no crea ejemplos independientes y puede sobreajustar los pequeños dominios.

In [ ]:
raw, cinic_info = base.download_cinic(CACHE_DIR)
dn, domain_info = m.download_domainnet(
    CACHE_DIR, download_dir=Path('/content/domainnet_downloads'), keep_archives=False)
data, data_identity = m.prepare_data(
    raw, dn, audit_x, audit_y, CFG, RUN_DIR,
    provenance={'cinic': cinic_info, 'domainnet': domain_info,
                'competition_validation_sha256': audit_sha})
del raw, dn, audit_x, audit_y
gc.collect()
counts = pd.DataFrame(data_identity['counts']).T
display(counts[['before', 'after']])
coverage = pd.DataFrame(
    {name: row['class_counts'] for name, row in data_identity['counts'].items()},
    index=m.CLASSES).T
display(coverage)
print('Claves de píxeles con etiquetas contradictorias excluidas del desarrollo:',
      data_identity['conflicting_pixel_keys'])


## 4. Ver una muestra pequeña del entrenamiento

Cada columna procede de una fuente de entrenamiento. No mostramos sketch ni imágenes de test. El azar tiene una semilla fija para que las figuras sean reproducibles.

El resize bilineal adapta la entrada al checkpoint S0 de **256×256**. No es superresolución y no recupera detalles ausentes en 32×32.

In [ ]:
rng = np.random.default_rng(2026)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for d, title in enumerate(('CINIC', 'DomainNet clipart', 'DomainNet painting')):
    pool = np.flatnonzero(data['domain'] == d)
    chosen = rng.choice(pool, 3, replace=False)
    for ax, i in zip(axes[:, d], chosen):
        ax.imshow(data['x'][i], interpolation='nearest')
        ax.set_title(f"{title}\n{m.CLASSES[int(data['y'][i])]}", fontsize=10)
        ax.axis('off')
plt.tight_layout(); plt.show()

chosen = rng.choice(np.flatnonzero(data['domain'] == 0), 4, replace=False)
fig, axes = plt.subplots(4, 2, figsize=(7, 12))
for row, i in enumerate(chosen):
    axes[row, 0].imshow(data['x'][i], interpolation='nearest')
    enlarged = Image.fromarray(data['x'][i]).resize((256, 256), Image.Resampling.BILINEAR)
    axes[row, 1].imshow(enlarged, interpolation='nearest')
    axes[row, 0].set_title(f"32×32 · {m.CLASSES[int(data['y'][i])]}")
    axes[row, 1].set_title('Bilineal → 256×256')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout(); plt.show()


## 5. Qué hace AdaIN y qué no hace

AdaIN utiliza un encoder VGG y un decoder **abiertos y congelados**. Cambia las medias y desviaciones de las características de una imagen de contenido tomando las de una imagen de estilo:

$$f_{nuevo}=\sigma_{estilo}\frac{f_{contenido}-\mu_{contenido}}{\sigma_{contenido}}+\mu_{estilo}.$$

Los donantes se toman **solo de painting-train después de deduplicar**, sin sus etiquetas y de un mismo banco para todas las clases. No se usan sketch, validación, test, un profesor clasificador ni prompts.

En esta primera implementación, todas las imágenes ya se representan a 32×32. Se amplían a 128×128 para ejecutar AdaIN, se reduce su salida de nuevo a 32×32 y se mezcla al 50% con el original para moderar la deformación. La mezcla de características usa alpha=0,5. Esto es una aproximación sobre imágenes pequeñas, **no una reproducción de Stylized-ImageNet** ni estilización de originales en alta resolución. El decoder puede introducir artefactos y alterar la legibilidad: las figuras permiten comprobarlo.

Se crea una vista fija por original, con emparejamientos reproducibles; C la utiliza con probabilidad 30%. Los bloques completos sobreviven a una interrupción. Se registra el coste de preparación aparte del entrenamiento.

In [ ]:
style_t0 = time.monotonic()
style_x, style_manifest = m.build_style_cache(
    data, CFG, STYLE_DIR, data_identity, session_hours=STYLE_HOURS)
print('Tiempo de esta llamada de preparación (min):', round((time.monotonic()-style_t0)/60, 1))
print('Cache de vistas:', style_x.shape, style_x.dtype)
base.atomic_json(dict(manifest=style_manifest, images=len(style_x),
                      current_call_seconds=time.monotonic()-style_t0),
                 RUN_DIR / 'style_summary.json')


In [ ]:
rng = np.random.default_rng(41)
# Una imagen por clase; incluye deer, ausente en DomainNet.
chosen = [rng.choice(np.flatnonzero((data['domain'] == 0) & (data['y'] == c))) for c in range(10)]
fig, axes = plt.subplots(10, 3, figsize=(9, 25))
for row, i in enumerate(chosen):
    axes[row, 0].imshow(data['x'][i], interpolation='nearest')
    axes[row, 1].imshow(style_x[i], interpolation='nearest')
    axes[row, 2].imshow(Image.fromarray(style_x[i]).resize((256,256), Image.Resampling.BILINEAR),
                       interpolation='nearest')
    axes[row, 0].set_ylabel(m.CLASSES[int(data['y'][i])])
    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
for ax, title in zip(axes[0], ('Original 32×32', 'AdaIN + mezcla, 32×32', 'Entrada ampliada 256×256')):
    ax.set_title(title)
plt.tight_layout(); plt.show()


## 6. Arquitectura y uso de las etiquetas

Una imagen aumentada a 32×32 se preprocesa con la referencia oficial de OpenCLIP: **256×256 bilineal y normalización 0/1**. El encoder visual S0 produce un vector de 512 componentes y una capa lineal entrenable produce diez logits.

El encoder S0 es híbrido, con convoluciones y atención. Se descarta la torre textual. Las etiquetas verdaderas de CINIC/DomainNet entrenan la cabeza y, después, el encoder mediante entropía cruzada y label smoothing 0,05. AdaIN únicamente produce vistas aumentadas; no suministra etiquetas.

Las tres variantes conservan exactamente el mismo número de parámetros de inferencia. VGG/decoder se utilizan solo al preparar datos y no forman parte del modelo final.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.4))
ax.set_xlim(0, 12); ax.set_ylim(0, 2); ax.axis('off')
boxes = [(1.2,'Imagen\n32×32 RGB'), (4.2,'Bilineal\n256×256'),
         (7.2,'S0 visual\nvector 512'), (10.2,'Linear 512→10\nlogits')]
for x, text in boxes:
    ax.text(x, 1, text, ha='center', va='center',
            bbox=dict(boxstyle='round,pad=.7', facecolor='#e8eef8', edgecolor='#42638d'))
for x in (2.1, 5.1, 8.1):
    ax.annotate('', xy=(x+1.2,1), xytext=(x,1), arrowprops=dict(arrowstyle='->'))
plt.show()


## 7. Entrenar A/B/C con el mismo presupuesto

Esta celda empieza cada variante desde los pesos originales, nunca desde la variante anterior. Verifica el hash de la inicialización completa. La selección usa las mismas tres particiones para todas.

**Reanudar:** mantén RUN_NAME y la configuración, vuelve a ejecutar las celdas anteriores y esta. Una interrupción a mitad de época repite esa época desde su checkpoint previo. El presupuesto de sesión se comprueba entre épocas; una primera época o una operación lenta puede sobrepasarlo. El tiempo de descarga y la preparación de estilos no están incluidos en SESSION_HOURS.

Después de la primera época se muestra una extrapolación basada en tiempo real; no se promete terminar las tres variantes en 6–8 horas.

In [ ]:
def initial_state_hash(model):
    h = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        h.update(name.encode()); h.update(tensor.detach().cpu().numpy().tobytes())
    return h.hexdigest()

PROTOCOLS = {}
session_deadline = time.monotonic() + SESSION_HOURS * 3600
for arm in m.ARMS:
    arm_dir = RUN_DIR / arm
    arm_dir.mkdir(parents=True, exist_ok=True)
    base.seed_all(CFG['seed'])
    model, train_tf, eval_tf, model_info = base.load_model(arm_dir, CFG)
    init = initial_state_hash(model)
    base.lock_json({'initial_state_sha256': init}, RUN_DIR / 'initial_state.json')
    protocol = dict(arm=arm, config=CFG, data_hash=base.json_hash(data_identity),
                    model=model_info, initial_state_sha256=init, code_sha256=CODE_HASHES,
                    environment=VERSIONS,
                    style=style_manifest if arm == m.ARMS[2] else None,
                    selection='equal mean of CINIC/clipart/painting macro accuracies',
                    audit='sketch and competition validation, after final choice')
    PROTOCOLS[arm] = protocol
    remaining = max(0., (session_deadline-time.monotonic())/3600)
    history = m.fit(model, data, train_tf, eval_tf, CFG, arm_dir, protocol, arm,
                    style_x=style_x if arm == m.ARMS[2] else None,
                    session_hours=remaining)
    if history:
        estimate = np.mean([r['epoch_seconds'] for r in history[-3:]])*CFG['max_epochs']/3600
        print(f'{arm}: extrapolación de una variante completa ~{estimate:.2f} h; cambia entre etapas.')
    del model
    gc.collect(); torch.cuda.empty_cache()
    status_path = arm_dir / 'status.json'
    if not status_path.exists() or not json.loads(status_path.read_text())['completed']:
        print('Estudio pausado. Reejecuta esta celda en la próxima sesión.')
        break


## 8. Comparación de selección y congelación del ganador

Se requiere haber completado las tres variantes. La regla se fijó antes de entrenar: mayor media de accuracy macro por dominio; desempate por menor pérdida y después por la variante más sencilla.

Los porcentajes de esta tabla son de selección externa. **No son scores de Kaggle.** Una mejora en un dominio no garantiza una mejora en otro.

In [ ]:
# Reconstruimos protocolos persistidos por si se ha reiniciado el runtime.
PROTOCOLS = {}
for arm in m.ARMS:
    p = RUN_DIR / arm / 'protocol.json'
    if not p.exists(): raise RuntimeError('Completa las tres variantes antes de comparar.')
    PROTOCOLS[arm] = json.loads(p.read_text())
chosen_method = m.choose_completed(RUN_DIR, PROTOCOLS)
rows = []
for arm in m.ARMS:
    status = json.loads((RUN_DIR / arm / 'status.json').read_text())
    b = status['best']
    rows.append(dict(method=arm, best_epoch=b['epoch'], selection_macro=b['accuracy'],
        cinic_macro=b['domains']['cinic']['macro_accuracy'],
        clipart_macro=b['domains']['clipart']['macro_accuracy'],
        painting_macro=b['domains']['painting']['macro_accuracy'],
        training_hours=status['training_hours'], updates=CFG['max_epochs']*CFG['steps_per_epoch']))
selection_table = pd.DataFrame(rows)
selection_table.to_csv(RUN_DIR / 'selection_results.csv', index=False)
display(selection_table.style.format({c:'{:.2%}' for c in
    ('selection_macro','cinic_macro','clipart_macro','painting_macro')}))
print('Método congelado antes de la evaluación reservada:', chosen_method['winner'])
fig, ax = plt.subplots(figsize=(9, 4))
for arm in m.ARMS:
    h = json.loads((RUN_DIR / arm / 'history.json').read_text())
    ax.plot([r['updates'] for r in h], [100*r['selection_accuracy'] for r in h], label=arm)
ax.set(xlabel='Actualizaciones acumuladas', ylabel='Media macro de selección (%)')
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.show()


## 9. Evaluación reservada: comprobar transferencia a otro dominio

Solo ahora calculamos métricas en sketch y la validación etiquetada de la competición. Evaluamos las tres variantes para el informe, pero **no cambiamos el ganador según estas métricas**.

Sketch solo cubre nueve clases y un estilo concreto; no sustituye al test de la competición. Si un método mejora selección y empeora aquí, es un resultado científico relevante, no una razón para ocultar la variante.

Se mantienen las diez salidas del clasificador incluso en la evaluación de nueve clases; no se elimina la salida deer ni se renormaliza el espacio de etiquetas.

In [ ]:
audit_record = {}
for arm in m.ARMS:
    arm_dir = RUN_DIR / arm
    base.seed_all(CFG['seed'])
    model, _, eval_tf, _ = base.load_model(arm_dir, CFG)
    base.load_selected(model, arm_dir, PROTOCOLS[arm])
    model.to('cuda')
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    audit_record[arm] = m.evaluate_groups(model, data['audit'], eval_tf, CFG, torch.device('cuda'), dtype)
    del model; gc.collect(); torch.cuda.empty_cache()
base.atomic_json(dict(chosen_before_audit=chosen_method, metrics=audit_record),
                 RUN_DIR / 'reserved_audit.json')
audit_table = pd.DataFrame([
    dict(method=arm, domain=domain, **{k:v for k,v in metrics.items() if k!='class_accuracy'})
    for arm, domains in audit_record.items() for domain, metrics in domains.items()])
audit_table.to_csv(RUN_DIR / 'reserved_audit.csv', index=False)
display(audit_table.style.format({'accuracy':'{:.2%}', 'macro_accuracy':'{:.2%}', 'loss':'{:.4f}'}))


## 10. Exportación opcional de un único CSV

Por defecto no se abre test.npz. Si decides utilizar el método ya congelado para la entrega, activa EXPORT_FINAL aquí. Se procesa la imagen completa con el preprocesado oficial; sin AdaIN, sin ensemble y sin ajustes a partir de test.

No hay submission automática. Se validan y alinean los IDs con sample_submission.csv.

In [ ]:
EXPORT_FINAL = False
if EXPORT_FINAL:
    chosen_method = m.choose_completed(RUN_DIR, PROTOCOLS)
    arm = chosen_method['winner']
    base.seed_all(CFG['seed'])
    model, _, eval_tf, _ = base.load_model(RUN_DIR / arm, CFG)
    model.to('cuda')
    csv_path = m.export_chosen(model, competition_root, eval_tf, CFG, RUN_DIR, PROTOCOLS)
    print('CSV:', csv_path)
    from google.colab import files
    files.download(str(csv_path))
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print('Exportación desactivada. Los resultados del estudio están guardados en Drive.')


## Para el informe

Reporta parámetros de inferencia, datos efectivos después de deduplicar, los mismos pasos para A/B/C, tiempo de preparación AdaIN y tiempo de entrenamiento, selección por dominio y evaluación reservada. No atribuyas a los datos una diferencia de GPU o de presupuesto. No compares esta A de 6.000 pasos con la ejecución antigua como si solo cambiara DomainNet.

Limitaciones: una semilla; pocas imágenes nuevas y sobremuestreo; ausencia de ciervos en DomainNet; cambios de textura imperfectos sobre 32×32; deduplicación exacta sin garantía frente a similares; selección sobre tres dominios que no representan todo lo posible. No hay garantía de alcanzar 97–98%.

Fuentes:
- [DomainNet: página oficial y listados depurados](https://ai.bu.edu/M3SDA/)
- [AdaIN: artículo](https://arxiv.org/abs/1703.06868) y [implementación abierta MIT](https://github.com/naoto0804/pytorch-AdaIN)
- [Stylized-ImageNet: motivación sobre forma/textura](https://arxiv.org/abs/1811.12231)
- [Checkpoint MobileCLIP2-S0 fijado](https://huggingface.co/timm/MobileCLIP2-S0-OpenCLIP/tree/095906d28bf54d7584dc411e8ffe448f34289e05)

El ZIP de entrega debe contener código/notebook sin outputs, datos, credenciales ni pesos; no incluyas la carpeta de resultados de Drive.